# 10 | CASALS–3DEP elevation comparison

This notebook reports raw `Refh − 3DEP` height differences and spatially held-out agreement after an empirical median offset for both available granules. H5 horizontal coordinates are treated as nominal WGS84 longitude/latitude and transformed in 2D to the LAS horizontal CRS (EPSG:6347). This does not establish a common vertical datum.

Only ASPRS class 2 points contribute to the reference surface. They are aggregated to 0.75 m cells, then sampled with a robust local plane inside a 3 m neighborhood. A point needs at least six neighbors. Slopes above 5° and plane residual NMAD above 0.20 m are flagged as locally unsuitable. There is no extrapolation beyond supported ground points.

The empirical offset uses one predeclared open-field site and is evaluated in a different open-field site 0.76–0.86 km away. Site membership is fixed before inspecting elevation differences. All H5 pulses are retained in raw diagnostics and stratified by SNR; sample selection never depends on the CASALS–3DEP residual.

**Strict absolute elevation accuracy remains unavailable** because the H5 vertical frame, LAS vertical CRS, and historical clip Z handling are not fully verified. Raw differences and empirically aligned holdout metrics are not absolute-accuracy certification.

In [1]:
from pathlib import Path
import json, sys
import h5py, laspy
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.path import Path as PolygonPath
from scipy.spatial import cKDTree

ROOT=Path.cwd()
if not (ROOT/'data').is_dir(): ROOT=ROOT.parent
if not (ROOT/'data').is_dir(): raise FileNotFoundError('Run from the CASALS project root.')
sys.path.insert(0,str(ROOT))
from research.elevation_comparison import horizontal_transform_audit,interpolate_ground_surface,elevation_metrics,spatial_site_masks,empirical_alignment_metrics
from research.l1b_validation import roi_primary_eligibility
H5_DIR=ROOT/'data/raw/casals_l1b'; LAZ_DIR=ROOT/'data/reference/3dep'
ROI_PATH=ROOT/'data/reference/landcover_rois/roi_manifest.geojson'
SIDECAR_DIR=ROOT/'Archive/backup/outputs/baseline_pre_refactor/l1b_outputs/download_3dep_lpc/clip_sidecars'
OUT_ROOT=ROOT/'outputs/notebooks/10_3dep_elevation_validation/revised'; OUT_ROOT.mkdir(parents=True,exist_ok=True)
H5_PATHS=sorted(H5_DIR.glob('casals_l1b_202411*T*_001_02.h5'))
if len(H5_PATHS)!=2: raise FileNotFoundError(f'Expected two H5 granules; found {len(H5_PATHS)}')
ROI_MANIFEST=json.loads(ROI_PATH.read_text(encoding='utf-8'))
NAIP_DATE_BY_GRANULE=ROI_MANIFEST['metadata']['naip_acquisition_dates_by_granule']
SPLIT_SITES={
 'casals_l1b_20241112T165718_001_02':{'calibration':'nov12_east_cropland_01','test':'nov12_southeast_cropland_02'},
 'casals_l1b_20241118T171757_001_02':{'calibration':'field_site_01','test':'field_site_02'},
}
GRID_RESOLUTION_M=0.75; SEARCH_RADIUS_M=3.0; MIN_NEIGHBORS=6; MAX_NEIGHBORS=32
MAX_SLOPE_DEG=5.0; MAX_PLANE_RESIDUAL_NMAD_M=0.20
H5_FIELDS=('refh_longitude','refh_latitude','refh','refh_snr','sweep_num','track_num','good_snr')

def json_safe(value):
 if isinstance(value,dict): return {str(k):json_safe(v) for k,v in value.items()}
 if isinstance(value,(list,tuple)): return [json_safe(v) for v in value]
 if isinstance(value,np.ndarray): return json_safe(value.tolist())
 if isinstance(value,np.bool_): return bool(value)
 if isinstance(value,np.integer): return int(value)
 if isinstance(value,(np.floating,float)): return float(value) if np.isfinite(value) else None
 if value is pd.NA or value is pd.NaT: return None
 if isinstance(value,Path): return str(value)
 return value

print({'h5_inputs':[p.name for p in H5_PATHS],'output_root':str(OUT_ROOT.relative_to(ROOT)),'split_sites':SPLIT_SITES})

{'h5_inputs': ['casals_l1b_20241112T165718_001_02.h5', 'casals_l1b_20241118T171757_001_02.h5'], 'output_root': 'outputs\\notebooks\\10_3dep_elevation_validation\\revised', 'split_sites': {'casals_l1b_20241112T165718_001_02': {'calibration': 'nov12_east_cropland_01', 'test': 'nov12_southeast_cropland_02'}, 'casals_l1b_20241118T171757_001_02': {'calibration': 'field_site_01', 'test': 'field_site_02'}}}


In [2]:
def build_class2_grid(laz_path,resolution_m=GRID_RESOLUTION_M):
 """Stream ASPRS class-2 points into 0.75 m cell means; do not load a whole LAZ."""
 with laspy.open(laz_path) as reader:
  header=reader.header; crs=header.parse_crs()
  if crs is None or crs.to_epsg()!=6347: raise RuntimeError(f'Expected EPSG:6347 LAS horizontal CRS, got {crs}')
  min_x,min_y=map(float,header.mins[:2]); max_x,max_y=map(float,header.maxs[:2])
  x0=np.floor(min_x/resolution_m)*resolution_m; y0=np.floor(min_y/resolution_m)*resolution_m
  nx=int(np.ceil((max_x-x0)/resolution_m))+1; ny=int(np.ceil((max_y-y0)/resolution_m))+1
  z_sum=np.zeros(nx*ny,dtype=np.float64); counts=np.zeros(nx*ny,dtype=np.int32); class2_n=0
  for chunk in reader.chunk_iterator(1_000_000):
   keep=np.asarray(chunk.classification)==2
   if not keep.any(): continue
   x=np.asarray(chunk.x[keep],dtype=np.float64); y=np.asarray(chunk.y[keep],dtype=np.float64); z=np.asarray(chunk.z[keep],dtype=np.float64)
   finite=np.isfinite(x)&np.isfinite(y)&np.isfinite(z); x,y,z=x[finite],y[finite],z[finite]
   ix=np.floor((x-x0)/resolution_m).astype(np.int64); iy=np.floor((y-y0)/resolution_m).astype(np.int64)
   inside=(ix>=0)&(ix<nx)&(iy>=0)&(iy<ny); flat=iy[inside]*nx+ix[inside]
   np.add.at(z_sum,flat,z[inside]); np.add.at(counts,flat,1); class2_n+=int(inside.sum())
 occupied=np.flatnonzero(counts); iy,ix=np.divmod(occupied,nx)
 ground_xy=np.column_stack((x0+(ix+0.5)*resolution_m,y0+(iy+0.5)*resolution_m))
 ground_z=z_sum[occupied]/counts[occupied]
 if not len(ground_z): raise RuntimeError('No finite ASPRS class-2 ground points were read.')
 tree=cKDTree(ground_xy,compact_nodes=True,balanced_tree=True)
 meta={'laz_path':str(laz_path.relative_to(ROOT)),'las_point_count':int(header.point_count),'las_header_bounds_m':{'min_x':float(header.mins[0]),'min_y':float(header.mins[1]),'min_z':float(header.mins[2]),'max_x':float(header.maxs[0]),'max_y':float(header.maxs[1]),'max_z':float(header.maxs[2])},'class2_point_count_streamed':class2_n,'class2_grid_cell_count':int(len(ground_z)),'grid_resolution_m':float(resolution_m),'grid_mean_points_per_occupied_cell':float(class2_n/len(ground_z)),'grid_max_points_per_cell':int(counts[occupied].max()),'crs':crs.to_string(),'vertical_axis_present':bool(any(a.direction.lower() in {'up','down'} for a in crs.axis_info[2:])),'class2_grid_z_min_m':float(ground_z.min()),'class2_grid_z_max_m':float(ground_z.max())}
 return ground_xy,ground_z,tree,meta


def assign_primary_open_field_sites(stem,longitude,latitude):
 features=[]; roi_audit=[]
 for feature in ROI_MANIFEST['features']:
  props=feature['properties']
  if props.get('granule_id')!=stem or props.get('class_label')!='open_field': continue
  audit=roi_primary_eligibility(props,expected_granule_id=stem,observed_image_date=NAIP_DATE_BY_GRANULE[stem])
  roi_audit.append(audit)
  if audit['primary_eligible']: features.append(feature)
 if len(features)!=2: raise RuntimeError(f'Expected two date-verified Included open-field sites for {stem}; found {len(features)}')
 expected=SPLIT_SITES[stem]; sites=[f['properties']['site_id'] for f in features]
 if set(sites)!={expected['calibration'],expected['test']}: raise RuntimeError(f'ROI sites {sites} do not match fixed split {expected}')
 site=np.full(longitude.size,'',dtype='U64'); roi=np.full(longitude.size,'',dtype='U64'); points=np.column_stack((longitude,latitude)); centers={}
 for feature in features:
  props=feature['properties']; coords=np.asarray(feature['geometry']['coordinates'][0],dtype=np.float64)
  inside=PolygonPath(coords).contains_points(points)
  if np.any(inside&(site!='')&(site!=props['site_id'])): raise RuntimeError('Primary open-field ROI polygons overlap.')
  assign=inside&(site==''); site[assign]=props['site_id']; roi[assign]=props['roi_id']
  center=coords[:-1].mean(axis=0); centers[props['site_id']]=(float(center[0]),float(center[1]))
 return site,roi,features,centers,roi_audit


def save_figure(fig,directory,filename):
 fig.savefig(directory/filename,dpi=180,bbox_inches='tight'); plt.close(fig)


In [3]:
def prepare_granule(h5_path):
 stem=h5_path.stem; laz_matches=sorted(LAZ_DIR.glob(f'{stem}_*.laz'))
 if len(laz_matches)!=1: raise FileNotFoundError(f'Expected one paired LAZ for {stem}; found {len(laz_matches)}')
 laz_path=laz_matches[0]; out_dir=OUT_ROOT/stem; out_dir.mkdir(parents=True,exist_ok=True)
 with h5py.File(h5_path,'r') as h5:
  scalar={name:np.asarray(h5[name]) for name in H5_FIELDS}; n=int(h5['refh'].shape[0])
  h5_attrs={str(k):str(v) for k,v in h5.attrs.items()}
 if any(v.size!=n for v in scalar.values()): raise RuntimeError('H5 scalar records have inconsistent lengths.')
 lon=scalar['refh_longitude']; lat=scalar['refh_latitude']; refh=scalar['refh']; snr=scalar['refh_snr']
 sweep=scalar['sweep_num'].astype(np.int32,copy=False); track=scalar['track_num'].astype(np.int16,copy=False)
 good_snr_consistent=bool(np.array_equal(scalar['good_snr'],snr>=5.0))
 geo_valid=np.isfinite(lon)&np.isfinite(lat); x=np.full(n,np.nan); y=np.full(n,np.nan)
 if not geo_valid.any(): raise RuntimeError('No finite H5 nominal coordinates.')
 xv,yv,transform_audit=horizontal_transform_audit(lon[geo_valid],lat[geo_valid],target_crs='EPSG:6347',area_of_interest=(float(lon[geo_valid].min()),float(lat[geo_valid].min()),float(lon[geo_valid].max()),float(lat[geo_valid].max())))
 x[geo_valid]=xv; y[geo_valid]=yv; xy=np.column_stack((x,y))
 base_valid=geo_valid&np.isfinite(refh)&np.isfinite(snr)
 site,roi,roi_features,site_centers,roi_audit=assign_primary_open_field_sites(stem,lon,lat)
 split_config=SPLIT_SITES[stem]; field_mask=site!=''; field_idx=np.flatnonzero(field_mask)
 cal_part,test_part=spatial_site_masks(site[field_idx],{split_config['calibration']})
 split=np.full(n,'outside_primary_open_field',dtype='U32'); split[field_idx[cal_part]]='calibration'; split[field_idx[test_part]]='independent_test'
 if set(np.unique(split[field_mask]))!={'calibration','independent_test'}: raise RuntimeError('Fixed calibration/test site split is incomplete.')
 ground_xy,ground_z,tree,grid_meta=build_class2_grid(laz_path)
 query_idx=np.flatnonzero(geo_valid)
 query_surface=interpolate_ground_surface(xy[query_idx],ground_xy,ground_z,search_radius_m=SEARCH_RADIUS_M,min_neighbors=MIN_NEIGHBORS,max_neighbors=MAX_NEIGHBORS,max_slope_deg=MAX_SLOPE_DEG,max_plane_residual_nmad_m=MAX_PLANE_RESIDUAL_NMAD_M,chunk_size=30000,ground_tree=tree)
 surface={}
 for key,values in query_surface.items():
  fill=np.nan if values.dtype.kind=='f' else (False if values.dtype.kind=='b' else (0 if values.dtype.kind in 'iu' else 'no_coordinate'))
  full=np.full(n,fill,dtype=values.dtype); full[query_idx]=values; surface[key]=full
 del query_surface
 print({'granule':stem,'h5_pulses':n,'finite_coordinates':int(geo_valid.sum()),'class2_points':grid_meta['class2_point_count_streamed'],'occupied_ground_cells':grid_meta['class2_grid_cell_count'],'plane_supported':int(surface['plane_fit_valid'].sum()),'acceptable_surface':int(surface['surface_acceptable'].sum()),'fixed_sites':split_config,'naip_date':NAIP_DATE_BY_GRANULE[stem]})
 return {'stem':stem,'h5_path':h5_path,'laz_path':laz_path,'out_dir':out_dir,'n':n,'h5_attrs':h5_attrs,'scalar':scalar,'lon':lon,'lat':lat,'refh':refh,'snr':snr,'sweep':sweep,'track':track,'x':x,'y':y,'xy':xy,'geo_valid':geo_valid,'base_valid':base_valid,'good_snr_consistent':good_snr_consistent,'site':site,'roi':roi,'split':split,'split_config':split_config,'roi_features':roi_features,'roi_audit':roi_audit,'site_centers':site_centers,'transform_audit':transform_audit,'ground_xy':ground_xy,'ground_z':ground_z,'tree':tree,'grid_meta':grid_meta,'surface':surface}


In [4]:
def analyze_granule(state):
 stem=state['stem']; out_dir=state['out_dir']; n=state['n']; snr=state['snr']; refh=state['refh']; site=state['site']; split=state['split']; surface=state['surface']
 plane=surface['reference_height_plane_m']; local_median=surface['reference_height_median_m']; supported=surface['plane_fit_valid']; acceptable=surface['surface_acceptable']
 base=state['base_valid']; raw=np.full(n,np.nan); raw_median=np.full(n,np.nan)
 good=base&supported; good_median=base&np.isfinite(local_median)
 raw[good]=refh[good]-plane[good]; raw_median[good_median]=refh[good_median]-local_median[good_median]
 strata={'all_valid_refh':base,'snr_ge_3':base&(snr>=3.0),'snr_ge_5':base&(snr>=5.0)}
 raw_rows=[]
 for population,pop_mask in [('whole_granule',base),('included_open_field',base&(site!=''))]:
  for quality,qmask in [('local_plane_supported',supported),('surface_acceptable',acceptable)]:
   for stratum_name,stratum_mask in strata.items():
    target=pop_mask&stratum_mask; take=target&qmask&np.isfinite(raw)
    row={'granule_id':stem,'population':population,'surface_quality':quality,'snr_stratum':stratum_name,'n_denominator':int(target.sum()),'coverage':float(take.sum()/target.sum()) if target.any() else None,'difference_definition':'CASALS H5 Refh minus local 3DEP class-2 plane height; vertical reference mismatch may be present','surface_method':'0.75 m cell mean plus robust local plane'}
    row.update(elevation_metrics(raw[take])); raw_rows.append(row)
    take_med=target&supported&np.isfinite(raw_median)
    row_med={'granule_id':stem,'population':population,'surface_quality':'local_median_check','snr_stratum':stratum_name,'n_denominator':int(target.sum()),'coverage':float(take_med.sum()/target.sum()) if target.any() else None,'difference_definition':'CASALS H5 Refh minus local 3DEP class-2 neighbor median; vertical reference mismatch may be present','surface_method':'local median over the same supported 3 m neighborhood'}
    row_med.update(elevation_metrics(raw_median[take_med])); raw_rows.append(row_med)
 pd.DataFrame(raw_rows).to_csv(out_dir/'elevation_raw_difference_metrics.csv',index=False)

 split_cfg=state['split_config']; alignment_rows=[]; holdout={}
 stratum_labels={'all_valid_refh':'All valid','snr_ge_3':'SNR ≥ 3','snr_ge_5':'SNR ≥ 5'}
 for name,mask in strata.items():
  cal=(split=='calibration')&acceptable&mask&np.isfinite(raw)
  test=(split=='independent_test')&acceptable&mask&np.isfinite(raw)
  denominator=(split=='independent_test')&mask
  if not cal.any() or not test.any(): raise RuntimeError(f'Empty supported calibration/test sample: {stem}, {name}')
  metrics=empirical_alignment_metrics(raw,cal,test,test_denominator=int(denominator.sum()))
  metrics.update({'granule_id':stem,'snr_stratum':name,'calibration_site_id':split_cfg['calibration'],'independent_test_site_id':split_cfg['test'],'split_rule':'whole predeclared site IDs, fixed before residual calculation','surface_acceptance':'at least 6 class-2 neighbors within 3 m; slope <=5 degrees; local plane residual NMAD <=0.20 m'})
  alignment_rows.append(metrics); resid=np.full(n,np.nan); resid[test]=raw[test]-metrics['empirical_vertical_offset_m']; holdout[stratum_labels[name]]=resid
 pd.DataFrame(alignment_rows).to_csv(out_dir/'elevation_empirical_holdout_metrics.csv',index=False)

 support_rows=[]
 regions=[('whole_granule',base)]+[(sid,base&(site==sid)) for sid in sorted(set(site)-{''})]
 for region,mask in regions:
  matched=mask&supported; accepted=mask&acceptable; nearest=surface['nearest_ground_distance_m'][mask]; nearest=nearest[np.isfinite(nearest)]
  row={'granule_id':stem,'region':region,'h5_valid_pulses':int(mask.sum()),'inside_3m_neighbor_radius':int((mask&(surface['neighbor_count']>0)).sum()),'at_least_6_neighbors':int((mask&(surface['neighbor_count']>=MIN_NEIGHBORS)).sum()),'plane_fit_valid':int(matched.sum()),'surface_acceptable':int(accepted.sum()),'surface_acceptable_fraction':float(accepted.sum()/mask.sum()) if mask.any() else None,'mean_robust_neighbor_count':float(np.mean(surface['robust_neighbor_count'][matched])) if matched.any() else None,'median_nearest_ground_distance_m':float(np.median(nearest)) if nearest.size else None}
  for key,threshold in [('snr_ge_3',3.0),('snr_ge_5',5.0)]:
   sm=mask&(snr>=threshold); row[f'{key}_h5_pulses']=int(sm.sum()); row[f'{key}_surface_acceptable']=int((sm&acceptable).sum())
  support_rows.append(row)
 pd.DataFrame(support_rows).to_csv(out_dir/'reference_ground_support.csv',index=False)

 from pyproj import Transformer
 transformer=Transformer.from_crs(4326,6347,always_xy=True); centers={sid:transformer.transform(*center) for sid,center in state['site_centers'].items()}
 a,b=split_cfg['calibration'],split_cfg['test']; spacing=float(np.hypot(centers[a][0]-centers[b][0],centers[a][1]-centers[b][1]))
 manifest=[]
 for sid,role in [(a,'calibration'),(b,'independent_test')]:
  m=site==sid; props=next(f['properties'] for f in state['roi_features'] if f['properties']['site_id']==sid)
  manifest.append({'granule_id':stem,'roi_id':props['roi_id'],'site_id':sid,'split':role,'class_label':'open_field','analysis_status':props['analysis_status'],'source_date':props['source_date'],'observed_naip_date':NAIP_DATE_BY_GRANULE[stem],'pulse_count':int(m.sum()),'plane_fit_valid_count':int((m&supported).sum()),'surface_acceptable_count':int((m&acceptable).sum()),'snr_ge_3_count':int((m&(snr>=3)).sum()),'snr_ge_5_count':int((m&(snr>=5)).sum()),'representative_site_x_m':centers[sid][0],'representative_site_y_m':centers[sid][1],'calibration_test_site_spacing_m':spacing,'fixed_before_residual_view':True,'geometry':'manual NAIP open-field polygon; nominal H5 pulse centers used for membership'})
 manifest.append({'granule_id':stem,'roi_id':'','site_id':'outside_primary_open_field','split':'not_in_primary_split','class_label':'not assigned','analysis_status':'not used for empirical offset','source_date':'','observed_naip_date':NAIP_DATE_BY_GRANULE[stem],'pulse_count':int((base&(site=='')).sum()),'plane_fit_valid_count':int((base&(site=='')&supported).sum()),'surface_acceptable_count':int((base&(site=='')&acceptable).sum()),'snr_ge_3_count':int((base&(site=='')&(snr>=3)).sum()),'snr_ge_5_count':int((base&(site=='')&(snr>=5)).sum()),'representative_site_x_m':None,'representative_site_y_m':None,'calibration_test_site_spacing_m':spacing,'fixed_before_residual_view':True,'geometry':'outside the two predeclared included open-field polygons'})
 pd.DataFrame(manifest).to_csv(out_dir/'spatial_calibration_test_manifest.csv',index=False)

 sample_idx=np.flatnonzero(base&supported)
 if sample_idx.size>12000: sample_idx=sample_idx[np.linspace(0,sample_idx.size-1,12000,dtype=int)]
 sensitivity=[]
 for label,dx,dy in [('east +2 m',2,0),('west -2 m',-2,0),('north +2 m',0,2),('south -2 m',0,-2)]:
  moved=interpolate_ground_surface(state['xy'][sample_idx]+np.array([dx,dy]),state['ground_xy'],state['ground_z'],search_radius_m=SEARCH_RADIUS_M,min_neighbors=MIN_NEIGHBORS,max_neighbors=MAX_NEIGHBORS,max_slope_deg=MAX_SLOPE_DEG,max_plane_residual_nmad_m=MAX_PLANE_RESIDUAL_NMAD_M,chunk_size=12000,ground_tree=state['tree'])
  delta=moved['reference_height_plane_m']-plane[sample_idx]; delta=np.abs(delta[np.isfinite(delta)])
  sensitivity.append({'granule_id':stem,'direction':label,'horizontal_shift_m':2.0,'query_count':int(sample_idx.size),'supported_perturbation_count':int(delta.size),'median_absolute_height_change_m':float(np.median(delta)) if delta.size else None,'p95_absolute_height_change_m':float(np.percentile(delta,95)) if delta.size else None,'max_absolute_height_change_m':float(delta.max()) if delta.size else None})
 pd.DataFrame(sensitivity).to_csv(out_dir/'xy_perturbation_sensitivity.csv',index=False)

 matched=supported; matched_idx=np.flatnonzero(matched)
 matched_df=pd.DataFrame({'pulse_index':matched_idx.astype(np.int32),'sweep_num':state['sweep'][matched],'track_num':state['track'][matched],'longitude_deg':state['lon'][matched],'latitude_deg':state['lat'][matched],'casals_refh_m':refh[matched],'refh_snr':snr[matched],'good_snr_flag':state['scalar']['good_snr'][matched],'easting_m_epsg6347':state['x'][matched],'northing_m_epsg6347':state['y'][matched],'reference_height_local_plane_m':plane[matched],'reference_height_local_median_m':local_median[matched],'raw_difference_plane_m':raw[matched],'raw_difference_local_median_m':raw_median[matched],'neighbor_count_within_3m':surface['neighbor_count'][matched],'robust_neighbor_count':surface['robust_neighbor_count'][matched],'nearest_ground_distance_m':surface['nearest_ground_distance_m'][matched],'local_slope_deg':surface['local_slope_deg'][matched],'plane_residual_nmad_m':surface['plane_residual_nmad_m'][matched],'ground_roughness_nmad_m':surface['ground_roughness_nmad_m'][matched],'surface_acceptable':acceptable[matched],'surface_status':pd.Categorical(surface['surface_status'][matched]),'primary_roi_id':pd.Categorical(state['roi'][matched]),'spatial_split':pd.Categorical(split[matched]),'empirical_holdout_residual_snr_ge_3_m':holdout['SNR ≥ 3'][matched]})
 matched_df.to_parquet(out_dir/'matched_casals_3dep_points.parquet',index=False,compression='zstd')

 laz_path=state['laz_path']; sidecar_path=SIDECAR_DIR/f'{laz_path.name}.json'; sidecar=json.loads(sidecar_path.read_text(encoding='utf-8')) if sidecar_path.exists() else None
 historical_provenance_evidence=None
 if sidecar is not None:
  from pyproj import Transformer
  plan=sidecar.get('clip_plan',{}); footprint=sidecar.get('casals_footprint',{}); clip_result=sidecar.get('clip_result',{})
  sidecar_source=sidecar.get('source_h5',''); footprint_source=footprint.get('source_h5','')
  bbox_text=plan.get('clip_local_bbox') or footprint.get('clip_local_bbox')
  bbox_delta=None; bbox_max_abs_delta=None; bbox_status='UNAVAILABLE'
  if bbox_text and plan.get('clip_crs') and state['grid_meta'].get('crs'):
   bbox_values=np.fromstring(str(bbox_text),sep=',')
   if bbox_values.size==4:
    xmin,ymin,xmax,ymax=map(float,bbox_values)
    transformer=Transformer.from_crs(plan['clip_crs'],state['grid_meta']['crs'],always_xy=True)
    corners=[transformer.transform(x,y) for x in (xmin,xmax) for y in (ymin,ymax)]
    expected={'min_x':min(x for x,y in corners),'max_x':max(x for x,y in corners),'min_y':min(y for x,y in corners),'max_y':max(y for x,y in corners)}
    header=state['grid_meta']['las_header_bounds_m']
    bbox_delta={key:float(expected[key]-header[key]) for key in expected}
    bbox_max_abs_delta=float(max(abs(v) for v in bbox_delta.values()))
    bbox_status='MATCH_WITHIN_1M' if bbox_max_abs_delta<=1.0 else 'REVIEW'
  historical_provenance_evidence={
   'sidecar_source_h5_basename':Path(sidecar_source).name if sidecar_source else None,
   'sidecar_footprint_source_h5_basename':Path(footprint_source).name if footprint_source else None,
   'source_h5_basename_matches_current_input':Path(sidecar_source).name.casefold()==state['h5_path'].name.casefold() if sidecar_source else False,
   'footprint_h5_basename_matches_current_input':Path(footprint_source).name.casefold()==state['h5_path'].name.casefold() if footprint_source else False,
   'clip_id':plan.get('clip_id'),'workunit':plan.get('workunit'),'project':plan.get('project'),
   'ept_url':plan.get('ept_url'),'ept_prefix':plan.get('ept_prefix'),
   'clip_crs':plan.get('clip_crs'),'output_crs':plan.get('output_crs_user_input'),
   'vertical_crs_context':plan.get('vert_crs'),'geoid_context':plan.get('geoid'),
   'vertical_datum_transform_applied':plan.get('vertical_datum_transform_applied'),
   'clip_result_status':clip_result.get('status'),'clip_result_point_count':clip_result.get('point_count'),
   'clip_result_point_count_matches_las_header':int(clip_result['point_count'])==state['grid_meta']['las_point_count'] if clip_result.get('point_count') is not None else False,
   'sidecar_output_basename_matches_current_laz':Path(plan.get('output_path','')).name.casefold()==laz_path.name.casefold(),
   'clip_local_bbox':bbox_text,'clip_local_bbox_to_las_header_delta_m':bbox_delta,
   'clip_local_bbox_horizontal_linkage_status':bbox_status,'clip_local_bbox_max_abs_delta_m':bbox_max_abs_delta,
   'meaning':'Horizontal footprint and file-count linkage only; this does not verify point-level Z preservation or vertical datum.'}
 audit={'granule_id':stem,'source_h5':str(state['h5_path'].relative_to(ROOT)),'source_laz':str(laz_path.relative_to(ROOT)),'source_h5_size_bytes':int(state['h5_path'].stat().st_size),'source_laz_size_bytes':int(laz_path.stat().st_size),'source_h5_attributes':state['h5_attrs'],'reference_grid':state['grid_meta'],'horizontal_transform':state['transform_audit'],'roi_eligibility':state['roi_audit'],'reference_frame_status':'UNRESOLVED','raw_height_comparison_status':'COMPLETED','empirical_alignment_status':'COMPLETED','strict_absolute_accuracy_status':'UNAVAILABLE','strict_unavailability_reasons':['H5 metadata does not verify the Refh vertical datum, realization, or coordinate epoch.','The paired LAZ has horizontal EPSG:6347 but no vertical CRS axis.','The historical sidecar reports EPSG:5703/GEOID18 context and vertical_datum_transform_applied=false, but that does not prove output Z was preserved from source EPT.','A complete verified XYZ vertical transformation is unavailable.'],'historical_sidecar_match':sidecar is not None,'historical_sidecar_path':str(sidecar_path.relative_to(ROOT)) if sidecar else None,'historical_provenance_evidence':historical_provenance_evidence,'historical_sidecar':sidecar,'source_ept_z_sample_check':'NOT_VERIFIED; no point-level source-to-clip Z comparison is claimed','historical_offsets':'Previously reported offsets near 37.7 m and 38.1 m were not reused; each offset was estimated from the current calibration samples.','interpretation_limit':'Raw and empirically aligned results are exploratory agreement only, not absolute elevation accuracy.'}
 (out_dir/'reference_frame_audit.json').write_text(json.dumps(json_safe(audit),indent=2,allow_nan=False),encoding='utf-8')
 return {'raw_rows':raw_rows,'alignment_rows':alignment_rows,'sensitivity':sensitivity,'holdout':holdout,'raw':raw,'raw_median':raw_median,'plane':plane,'local_median':local_median,'supported':supported,'acceptable':acceptable,'matched_count':int(matched.sum()),'audit':audit}


In [5]:
def write_elevation_figures(state,result):
 stem=state['stem']; out=state['out_dir']; n=state['n']; snr=state['snr']; raw=result['raw']; acceptable=result['acceptable']; supported=result['supported']; site=state['site']; track=state['track']
 stride=max(1,n//50000); sample=np.arange(0,n,stride)
 fig,ax=plt.subplots(figsize=(11,8),constrained_layout=True)
 status=np.where(acceptable[sample],1,np.where(supported[sample],0,-1))
 sc=ax.scatter(state['x'][sample],state['y'][sample],c=status,s=2,cmap='RdYlGn',vmin=-1,vmax=1,rasterized=True)
 ax.set(title=f'3DEP class-2 support | {stem}',xlabel='Easting (m, EPSG:6347)',ylabel='Northing (m, EPSG:6347)')
 cb=fig.colorbar(sc,ax=ax,ticks=[-1,0,1]); cb.ax.set_yticklabels(['no plane support','supported, quality flagged','acceptable surface'])
 save_figure(fig,out,'ground_support_map.png')

 fig,ax=plt.subplots(figsize=(10,6),constrained_layout=True)
 for label,mask in [('All valid',np.ones(n,dtype=bool)),('SNR ≥ 3',snr>=3),('SNR ≥ 5',snr>=5)]:
  vals=raw[acceptable&mask&np.isfinite(raw)]
  if vals.size:
   if vals.size>250000: vals=vals[np.linspace(0,vals.size-1,250000,dtype=int)]
   lo,hi=np.percentile(vals,[0.5,99.5]); ax.hist(vals,bins=100,range=(lo,hi),density=True,histtype='step',lw=1.8,label=f'{label} (n={int((acceptable&mask&np.isfinite(raw)).sum()):,})')
 ax.axvline(0,color='black',lw=1,ls='--'); ax.set(title='Raw height differences on acceptable local surfaces',xlabel='CASALS Refh − 3DEP plane height (m; datum mismatch may remain)',ylabel='Density')
 ax.text(.99,.02,'Display range: p0.5–p99.5; metrics use every matched point.',transform=ax.transAxes,ha='right',fontsize=8); ax.legend()
 save_figure(fig,out,'raw_difference_distribution.png')

 fig,ax=plt.subplots(figsize=(10,6),constrained_layout=True)
 for label,values in result['holdout'].items():
  values=values[np.isfinite(values)]
  if values.size: ax.hist(values,bins=80,density=True,histtype='step',lw=1.8,label=f'{label} (n={values.size:,})')
 ax.axvline(0,color='black',lw=1,ls='--'); ax.set(title='Spatial holdout residuals after empirical median alignment',xlabel='Holdout residual (m; not absolute accuracy)',ylabel='Density'); ax.legend()
 save_figure(fig,out,'empirical_holdout_residual_distribution.png')

 test=(site==state['split_config']['test'])&acceptable&(snr>=3)&np.isfinite(result['holdout']['SNR ≥ 3']); idx=np.flatnonzero(test)
 fig,ax=plt.subplots(figsize=(11,7),constrained_layout=True)
 if idx.size:
  values=result['holdout']['SNR ≥ 3'][idx]; lim=max(float(np.percentile(np.abs(values),98)),0.05)
  sc=ax.scatter(state['x'][idx],state['y'][idx],c=values,s=5,cmap='RdBu_r',norm=TwoSlopeNorm(vmin=-lim,vcenter=0,vmax=lim),rasterized=True); fig.colorbar(sc,ax=ax,label='Empirically aligned test residual (m)')
 ax.set(title='Independent open-field test-site residuals (SNR ≥ 3)',xlabel='Easting (m, EPSG:6347)',ylabel='Northing (m, EPSG:6347)')
 save_figure(fig,out,'empirical_holdout_spatial_map.png')

 idx=np.flatnonzero(acceptable&np.isfinite(raw)); idx=idx[::max(1,idx.size//120000)]
 fig,ax=plt.subplots(figsize=(10,6),constrained_layout=True)
 hb=ax.hexbin(snr[idx],raw[idx],gridsize=65,mincnt=1,bins='log',cmap='viridis'); ax.axhline(0,color='white',lw=1,ls='--')
 ax.set(title='Raw difference versus native H5 SNR',xlabel='H5 Refh SNR',ylabel='CASALS Refh − 3DEP plane height (m)'); fig.colorbar(hb,ax=ax,label='log10 pulse count')
 save_figure(fig,out,'raw_difference_vs_snr.png')

 fig,ax=plt.subplots(figsize=(11,6),constrained_layout=True)
 for label,mask,color in [('All valid',np.ones(n,dtype=bool),'#777777'),('SNR ≥ 3',snr>=3,'#2b75a1'),('SNR ≥ 5',snr>=5,'#d2691e')]:
  vals=pd.Series(raw[mask&acceptable&np.isfinite(raw)]).groupby(track[mask&acceptable&np.isfinite(raw)]).median()
  ax.plot(vals.index,vals.values,lw=1.3,label=label,color=color)
 ax.axhline(0,color='black',lw=1,ls='--'); ax.set(title='Median raw difference by track',xlabel='Track index',ylabel='CASALS Refh − 3DEP plane height (m)'); ax.legend()
 save_figure(fig,out,'across_track_raw_differences.png')

 primary=(site!='')&acceptable&np.isfinite(state['refh'])&np.isfinite(result['plane']); counts=np.bincount(track[primary].astype(int),minlength=256); chosen=int(np.argmax(counts)) if counts.any() else -1
 idx=np.flatnonzero(primary&(track==chosen)) if chosen>=0 else np.array([],dtype=int); idx=idx[np.argsort(state['sweep'][idx])]
 if idx.size>25000: idx=idx[::int(np.ceil(idx.size/25000))]
 fig,ax=plt.subplots(figsize=(11,6),constrained_layout=True)
 if idx.size:
  ax.plot(state['sweep'][idx],state['refh'][idx],lw=.8,label='CASALS H5 Refh')
  ax.plot(state['sweep'][idx],result['plane'][idx],lw=.8,label='3DEP local plane')
 ax.set(title=f'Open-field reference transect | track {chosen}',xlabel='Sweep index',ylabel='Stored height (m; raw vertical offset retained)'); ax.legend()
 save_figure(fig,out,'open_field_ground_transect.png')

 fig,ax=plt.subplots(figsize=(9,5),constrained_layout=True); table=pd.DataFrame(result['sensitivity'])
 ax.bar(table['direction'],table['p95_absolute_height_change_m'],color='#4a90a4'); ax.set(title='Sensitivity to ±2 m horizontal perturbations',ylabel='P95 |3DEP height change| (m)',xlabel='Perturbation direction')
 save_figure(fig,out,'xy_perturbation_sensitivity.png')

 return ['ground_support_map.png','raw_difference_distribution.png','empirical_holdout_residual_distribution.png','empirical_holdout_spatial_map.png','raw_difference_vs_snr.png','across_track_raw_differences.png','open_field_ground_transect.png','xy_perturbation_sensitivity.png']


In [6]:
import gc
RUN_SUMMARIES=[]
for h5_path in H5_PATHS:
 state=prepare_granule(h5_path)
 result=analyze_granule(state)
 figures=write_elevation_figures(state,result)
 summary={
  'granule_id':state['stem'],'input_pulses':state['n'],'valid_h5_pulses':int(state['base_valid'].sum()),
  'horizontal_transform':state['transform_audit'],'reference_surface':state['grid_meta'],'historical_provenance_evidence':result['audit']['historical_provenance_evidence'],
  'good_snr_equals_snr_ge_5':state['good_snr_consistent'],'raw_difference_metrics':result['raw_rows'],
  'empirical_holdout_metrics':result['alignment_rows'],'xy_perturbation_sensitivity':result['sensitivity'],
  'reference_frame_status':'UNRESOLVED','raw_height_comparison_status':'COMPLETED',
  'empirical_alignment_status':'COMPLETED','strict_absolute_accuracy_status':'UNAVAILABLE',
  'outputs':['reference_frame_audit.json','reference_ground_support.csv','matched_casals_3dep_points.parquet','spatial_calibration_test_manifest.csv','elevation_raw_difference_metrics.csv','elevation_empirical_holdout_metrics.csv','elevation_metrics_summary.json','xy_perturbation_sensitivity.csv']+figures,
 }
 (state['out_dir']/'elevation_metrics_summary.json').write_text(json.dumps(json_safe(summary),indent=2,allow_nan=False),encoding='utf-8')
 RUN_SUMMARIES.append(summary)
 print(json.dumps(json_safe({'granule':state['stem'],'class2_points':state['grid_meta']['class2_point_count_streamed'],'matched_pulses':result['matched_count'],'acceptable_surfaces':int(result['acceptable'].sum()),'raw_whole_granule_all_valid':[r for r in result['raw_rows'] if r['population']=='whole_granule' and r['surface_quality']=='surface_acceptable' and r['snr_stratum']=='all_valid_refh'],'empirical_holdout':result['alignment_rows'],'strict_absolute_accuracy_status':'UNAVAILABLE','output_dir':str(state['out_dir'].relative_to(ROOT))}),indent=2))
 del state,result
 gc.collect()
print(f'Completed actual raw and spatially held-out CASALS–3DEP comparisons for {len(RUN_SUMMARIES)} paired granules.')
summary_rows=[]
for s in RUN_SUMMARIES:
 raw=next(r for r in s['raw_difference_metrics'] if r['population']=='whole_granule' and r['surface_quality']=='surface_acceptable' and r['snr_stratum']=='all_valid_refh')
 main=next(r for r in s['empirical_holdout_metrics'] if r['snr_stratum']=='snr_ge_3')
 summary_rows.append({'granule':s['granule_id'],'raw matched n':raw['n'],'raw median (m)':raw['median_m'],'raw RMSE (m)':raw['rmse_m'],'SNR≥3 holdout n':main['test_n'],'SNR≥3 offset (m)':main['empirical_vertical_offset_m'],'SNR≥3 test RMSE (m)':main['rmse_m'],'strict absolute status':s['strict_absolute_accuracy_status']})
display(pd.DataFrame(summary_rows))

C:\Users\zhong267\OneDrive - purdue.edu\CASALS\research\elevation_comparison.py:176: RuntimeWarning: All-NaN slice encountered
  residual_center = np.nanmedian(residual, axis=1)
C:\Users\zhong267\OneDrive - purdue.edu\CASALS\research\elevation_comparison.py:177: RuntimeWarning: All-NaN slice encountered
  initial_nmad = 1.4826 * np.nanmedian(np.abs(residual - residual_center[:, None]), axis=1)


{'granule': 'casals_l1b_20241112T165718_001_02', 'h5_pulses': 3604480, 'finite_coordinates': 3604480, 'class2_points': 4646895, 'occupied_ground_cells': 2382632, 'plane_supported': 3559190, 'acceptable_surface': 3369562, 'fixed_sites': {'calibration': 'nov12_east_cropland_01', 'test': 'nov12_southeast_cropland_02'}, 'naip_date': '2023-06-02'}


{
  "granule": "casals_l1b_20241112T165718_001_02",
  "class2_points": 4646895,
  "matched_pulses": 3559190,
  "acceptable_surfaces": 3369562,
  "raw_whole_granule_all_valid": [
    {
      "granule_id": "casals_l1b_20241112T165718_001_02",
      "population": "whole_granule",
      "surface_quality": "surface_acceptable",
      "snr_stratum": "all_valid_refh",
      "n_denominator": 3604480,
      "coverage": 0.9348261052911931,
      "difference_definition": "CASALS H5 Refh minus local 3DEP class-2 plane height; vertical reference mismatch may be present",
      "surface_method": "0.75 m cell mean plus robust local plane",
      "n": 3369562,
      "bias_m": -12.164465287251764,
      "median_m": -37.57069154902173,
      "std_m": 92.94880033297837,
      "rmse_m": 93.74140565986637,
      "mae_m": 75.18142562390983,
      "nmad_m": 43.23690122638924,
      "p05_m": -166.3108345007521,
      "p50_m": -37.57069154902173,
      "p95_m": 173.57868255276802,
      "absolute_p90_m": 170.0

C:\Users\zhong267\OneDrive - purdue.edu\CASALS\research\elevation_comparison.py:176: RuntimeWarning: All-NaN slice encountered
  residual_center = np.nanmedian(residual, axis=1)
C:\Users\zhong267\OneDrive - purdue.edu\CASALS\research\elevation_comparison.py:177: RuntimeWarning: All-NaN slice encountered
  initial_nmad = 1.4826 * np.nanmedian(np.abs(residual - residual_center[:, None]), axis=1)


{'granule': 'casals_l1b_20241118T171757_001_02', 'h5_pulses': 3604480, 'finite_coordinates': 3604480, 'class2_points': 63226508, 'occupied_ground_cells': 6959693, 'plane_supported': 3598281, 'acceptable_surface': 3457681, 'fixed_sites': {'calibration': 'field_site_01', 'test': 'field_site_02'}, 'naip_date': '2022-06-15'}


{
  "granule": "casals_l1b_20241118T171757_001_02",
  "class2_points": 63226508,
  "matched_pulses": 3598281,
  "acceptable_surfaces": 3457681,
  "raw_whole_granule_all_valid": [
    {
      "granule_id": "casals_l1b_20241118T171757_001_02",
      "population": "whole_granule",
      "surface_quality": "surface_acceptable",
      "snr_stratum": "all_valid_refh",
      "n_denominator": 3604480,
      "coverage": 0.9592731822620738,
      "difference_definition": "CASALS H5 Refh minus local 3DEP class-2 plane height; vertical reference mismatch may be present",
      "surface_method": "0.75 m cell mean plus robust local plane",
      "n": 3457681,
      "bias_m": -31.296549580003212,
      "median_m": -37.96164764544714,
      "std_m": 32.71284630718158,
      "rmse_m": 45.27255260792385,
      "mae_m": 38.922783837842054,
      "nmad_m": 0.22745610833488358,
      "p05_m": -38.23956944058026,
      "p50_m": -37.96164764544714,
      "p95_m": -15.186396114252867,
      "absolute_p90_m": 

,granule,raw matched n,raw median (m),raw RMSE (m),SNR≥3 holdout n,SNR≥3 offset (m),SNR≥3 test RMSE (m),strict absolute status
0,casals_l1b_20241112T165718_001_02,3369562,-37.570692,93.741406,4274,-37.659448,8.521722,UNAVAILABLE
1,casals_l1b_20241118T171757_001_02,3457681,-37.961648,45.272553,8623,-37.981134,0.468919,UNAVAILABLE


## Interpretation

The raw metrics preserve the measured vertical offset and may contain a datum mismatch. Each empirical offset is estimated independently per granule and SNR stratum from one calibration site; the other site is held out for scoring. These holdout residuals describe agreement after empirical alignment, not absolute accuracy.

The selected 2D horizontal operation and its stated accuracy are recorded. A ±2 m XY perturbation sensitivity is included because horizontal uncertainty changes the terrain query location. The local median provides a non-parametric interpolation check using the same supported class-2 neighborhood. The strict status remains `UNAVAILABLE` until the vertical frames, epoch, and historical clip Z path are directly verified.